# Assignment 1 — Words as Vectors: *Count* vs. *Predict*

**Available:** Saturday, **November 7, 2026, 09:00**  **Deadline:** Friday, **November 20, 2026, 23:00** (submit via **Stud.IP**)

> *"You shall know a word by the company it keeps."* — J.R. Firth (1957)

In the lecture *How to represent words* we saw two families of distributional word vectors:

| | **Count-based** | **Predictive (neural)** |
|---|---|---|
| Idea | count which context words appear around a target word | train a model to *predict* words from their context (or score windows) |
| Example | co-occurrence matrix + PMI (PPMI) | **CBOW** (Mikolov et al. 2013), **C&W** (Collobert et al. 2011) |
| Output | an embedding matrix $\mathbf{E} \in \mathbb{R}^{|V| \times |C|}$ | an embedding matrix $\mathbf{E} \in \mathbb{R}^{|V| \times d}$ |

In this assignment you will **build both from scratch on the same corpus**, then put them in a ring and let them fight:
nearest neighbours, correlation with **human similarity judgements**, and the famous **king − man + woman = ?** analogies.
At the end you decide: *don't count, predict*, or not?

### What you will learn
1. How to turn raw text into a co-occurrence matrix, and why **PMI** makes the counts much more useful.
2. How a neural embedding model is just a **lookup table $\mathbf{E}$ + a score function + a loss**, exactly as on the slides.
3. How to write the **forward pass** and the **loss** of CBOW and of the C&W ranking model in PyTorch.
4. How we **evaluate** word vectors, and how to read the results critically.

### Practical information
- Work in **groups of 1 to 3 students** (groups of 3 are encouraged). Join a group on Stud.IP by **November 20, 23:00**.
- Submit **one notebook per group** on Stud.IP, **with all outputs visible** (run everything before saving).
- **Do not enter your names inside the notebook.** Name the notebook file you upload to Stud.IP using the **surname(s) of all group members** (e.g. `A1_Smith_Miller.ipynb`).
- **Points:** 100 in total. Code tasks are marked 💻, written questions are marked ✍️. Write answers in your own words; 2–5 sentences is usually enough.
  - Part 0: 5 points · Part 1: 7 points · Part 2: 30 points · Part 3: 33 points · Part 4: 25 points.
  - **Total: 100 points.** Task 2.4 is assessed as Task 4.1 and its 5 points are included in Part 4 (not counted twice).
- **Runtime:** we recommend **Google Colab with a GPU** (`Runtime → Change runtime type → T4 GPU`). On a GPU the whole notebook runs in about 10 minutes. It also works on a laptop CPU (about 10–15 minutes), automatically using a smaller corpus.
- You only need `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib` and `torch` (all preinstalled on Colab).
- Cells marked **(given)** are complete: just run them. Cells with `# TODO` are yours.
- Small **✅ check** cells test your code. Passing them is a good sign, but not a guarantee of full points.

## Setup (given)

In [7]:
import os, gzip, math, time, urllib.request, collections
import numpy as np
import scipy.sparse as sp
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", DEVICE)

Running on: cpu


In [8]:
# Global settings (given). You may change them for experiments, but submit with the defaults.
N_TOKENS = 10_000_000 if DEVICE == "cuda" else 2_000_000   # size of the corpus we use
V_SIZE   = 10_000      # vocabulary: the V most frequent words (targets AND contexts)
WINDOW   = 2           # w: words to the left and to the right (as on the slides: w_{n-2} ... w_{n+2})
DIM      = 100         # dimensionality d of the predictive vectors (CBOW, C&W)
CBOW_EPOCHS = 2
CW_EPOCHS   = 3
BATCH_SIZE  = 1024

In [9]:
# Download the data (given). If a download fails, get the files from Stud.IP and put them in ./data
DATA = "data"; os.makedirs(DATA, exist_ok=True)
FILES = {
    "text8.gz":            "https://github.com/RaRe-Technologies/gensim-data/releases/download/text8/text8.gz",
    "men.csv":             "https://raw.githubusercontent.com/vecto-ai/word-benchmarks/master/word-similarity/monolingual/en/men.csv",
    "simlex999.txt":       "https://raw.githubusercontent.com/piskvorky/gensim/develop/gensim/test/test_data/simlex999.txt",
    "wordsim353.tsv":      "https://raw.githubusercontent.com/piskvorky/gensim/develop/gensim/test/test_data/wordsim353.tsv",
    "questions-words.txt": "https://raw.githubusercontent.com/piskvorky/gensim/develop/gensim/test/test_data/questions-words.txt",
}
for fname, url in FILES.items():
    path = os.path.join(DATA, fname)
    if not os.path.exists(path):
        print("downloading", fname, "...")
        urllib.request.urlretrieve(url, path)
print("all files ready:", sorted(os.listdir(DATA)))

downloading text8.gz ...
downloading men.csv ...
downloading simlex999.txt ...
downloading wordsim353.tsv ...
downloading questions-words.txt ...
all files ready: ['men.csv', 'questions-words.txt', 'simlex999.txt', 'text8.gz', 'wordsim353.tsv']


---
## Part 0 — Math warm-up (5 points)

> 🧭 **Foundations box.** Everything in this assignment uses only a handful of ideas. If they are fresh for you, read this carefully; if not, skim it.
>
> **Vectors.** A word vector is a list of numbers, e.g. $\mathbf{u} = [0, 1, 0, 2]$. The **dot product** multiplies entry by entry and sums: $\mathbf{u}\cdot\mathbf{v} = \sum_i u_i v_i$. Large when both vectors are large in the *same* dimensions.
>
> **Length (norm).** $\|\mathbf{u}\| = \sqrt{\sum_i u_i^2}$ — Pythagoras in many dimensions.
>
> **Cosine similarity.** $\cos(\mathbf{u},\mathbf{v}) = \dfrac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}$ is the cosine of the angle between the vectors: $1$ = same direction, $0$ = orthogonal (nothing in common), $-1$ = opposite. Dividing by the lengths makes it **norm-invariant**: only the *direction* matters.
>
> **Matrices and one-hot lookup.** An embedding matrix $\mathbf{E}$ has one row per word. A one-hot vector has a single 1 at the word's index, so $\text{onehot}_{cat}^\top \mathbf{E}$ simply **selects the row** of *cat*. In code we never build one-hot vectors, we just index: `E[word2id["cat"]]`.
>
> **Probabilities and logs.** $P(x)\in[0,1]$, and probabilities over all outcomes sum to 1. $\log$ turns products into sums ($\log ab = \log a + \log b$) and is negative for numbers in $(0,1)$. $\log 1 = 0$.
>
> **Softmax.** Turns any vector of real-valued scores $\mathbf{l}$ into probabilities: $\text{softmax}(\mathbf{l})_i = \dfrac{e^{l_i}}{\sum_j e^{l_j}}$. Bigger score ⇒ bigger probability; all outputs positive; they sum to 1.

### 💻 Task 0.1 (3 pts) — cosine similarity
Implement `cosine(u, v)` with numpy (no library similarity functions). Then check that you reproduce the numbers from the lecture slides for the toy vectors over the basis vocabulary *{bit, cute, furry, loud, miaowed, purred, ran, small}*: $\cos(\text{kitten},\text{cat}) \approx 0.58$, $\cos(\text{kitten},\text{dog}) = 0.00$, $\cos(\text{cat},\text{dog}) \approx 0.29$.

In [10]:
def cosine(u, v):
    # TODO: return the cosine similarity between the 1-D numpy arrays u and v
    raise NotImplementedError

In [11]:
# ✅ check (given)
kitten = np.array([0, 1, 0, 0, 1, 1, 0, 1])
cat    = np.array([0, 1, 1, 0, 1, 0, 0, 0])
dog    = np.array([1, 0, 1, 1, 0, 0, 1, 0])
for a, b, name in [(kitten, cat, "kitten-cat"), (kitten, dog, "kitten-dog"), (cat, dog, "cat-dog")]:
    print(f"{name}: {cosine(a, b):.2f}")
assert abs(cosine(kitten, cat) - 0.577) < 1e-2 and abs(cosine(cat, dog) - 0.289) < 1e-2
assert abs(cosine(2 * kitten, cat) - cosine(kitten, cat)) < 1e-9, "cosine should be norm-invariant"
print("✅ cosine works")

NotImplementedError: 

### ✍️ Question 0.2 (2 pts)
Imagine two count vectors: *the* (seen 1,000,000 times) and *a* (seen 500,000 times), with almost identical context *proportions*. Their dot product would be huge and the dot product of *a* with a rare word tiny. Why is **cosine** a better choice than the plain dot product for comparing count vectors?

**Your answer:**

*(write here)*

---
## Part 1 — Corpus and vocabulary (7 points)

We use **text8**: the first 100 MB of an English Wikipedia dump, already lower-cased and stripped of punctuation and numbers (so no preprocessing headaches this time). We take the first `N_TOKENS` tokens.

In [ ]:
# (given) load the first N_TOKENS tokens
with gzip.open(os.path.join(DATA, "text8.gz"), "rt") as f:
    text = f.read()
tokens = text.split(maxsplit=N_TOKENS)[:N_TOKENS]
del text
print(f"{len(tokens):,} tokens; first 30:\n", " ".join(tokens[:30]))

### 💻 Task 1.1 (5 pts) — vocabulary and integer ids
Computers prefer numbers to strings. Build:
- `word_counts`: a `collections.Counter` of all tokens,
- `id2word`: a list of the `V_SIZE` most frequent words, most frequent first,
- `word2id`: a dict mapping each of these words to its index in `id2word`,
- `ids`: a numpy array (`dtype=np.int64`) with the id of every token in `tokens` **that is in the vocabulary**. Tokens outside the vocabulary are simply dropped (a common simplification, also used by word2vec).

In [ ]:
# TODO
word_counts = None
id2word = None
word2id = None
ids = None

In [ ]:
# ✅ check (given)
assert len(id2word) == V_SIZE and word2id[id2word[123]] == 123
assert id2word[0] == "the" and ids.dtype == np.int64 and ids.max() < V_SIZE
coverage = len(ids) / len(tokens)
print(f"vocabulary covers {coverage:.1%} of the tokens; {len(ids):,} tokens kept")
print("10 most frequent:", id2word[:10])
print("10 least frequent in vocab:", id2word[-10:], "(count:", word_counts[id2word[-1]], ")")

### ✍️ Question 1.2 (2 pts)
Only 10,000 word *types* (out of more than 100,000 in the corpus) cover the large majority of the *tokens*. Which statistical law of language is behind this? Name one disadvantage of dropping all out-of-vocabulary tokens before building our windows.

**Your answer:**

*(write here)*

---
## Part 2 — The *count* approach (30 points)

Recipe from the slides:
1. Define a **basis vocabulary** $C$ of context words (here: the same `V_SIZE` words as our targets).
2. Define a **window** size $w$ (`WINDOW = 2`).
3. **Count** the basis words occurring up to $w$ words to the left or right of each occurrence of a target word.
4. The row of counts is the **vector** of the target word.

The result is a $|V| \times |C|$ matrix: 10,000 × 10,000 = 100 million cells, but most are 0. We therefore store it as a **sparse matrix** (only the non-zero cells are kept), using `scipy.sparse`.

### 💻 Task 2.1 (11 pts) — co-occurrence counts
Implement `count_cooccurrences(ids, V, window)` returning a `scipy.sparse.csr_matrix` `M` of shape `(V, V)` where `M[t, c]` is how often context word `c` appears within `window` positions of target word `t`.

**Hint (fast, no Python loop over tokens).** For a distance `d` (1, 2, …, `window`), the pairs of tokens that are `d` apart are simply `(ids[:-d], ids[d:])`: each word and the word `d` to its right. The window is symmetric, so also add the reverse pairs `(ids[d:], ids[:-d])`. Collect all row indices and column indices, and build `sp.coo_matrix((data, (rows, cols)), shape=(V, V))` with `data` all ones: **duplicate (row, col) entries are summed automatically** when you convert with `.tocsr()`.

In [ ]:
def count_cooccurrences(ids, V, window):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ check on a toy corpus (given)
toy = "the cute kitten purred the cute furry cat purred".split()
toy_vocab = sorted(set(toy)); toy_w2i = {w: i for i, w in enumerate(toy_vocab)}
toy_ids = np.array([toy_w2i[w] for w in toy])
M_toy = count_cooccurrences(toy_ids, len(toy_vocab), window=2)
print(pd.DataFrame(M_toy.toarray().astype(int), index=toy_vocab, columns=toy_vocab))
assert (M_toy != M_toy.T).nnz == 0, "a symmetric window must give a symmetric matrix"
assert M_toy[toy_w2i["kitten"], toy_w2i["cute"]] == 1 and M_toy[toy_w2i["the"], toy_w2i["cute"]] == 2
assert M_toy[toy_w2i["purred"], toy_w2i["the"]] == 1 and M_toy[toy_w2i["cat"], toy_w2i["kitten"]] == 0
assert M_toy.sum() == 2 * (2 * len(toy) - 3)
print("✅ counts look right")

In [ ]:
# (given) the real thing
t0 = time.time()
M_counts = count_cooccurrences(ids, V_SIZE, WINDOW)
print(f"done in {time.time()-t0:.1f}s; non-zero cells: {M_counts.nnz:,} ({M_counts.nnz / V_SIZE**2:.1%} of the matrix)")

### Not all counts are equal: Pointwise Mutual Information
*the* co-occurs with almost every word, so raw counts are dominated by **independently frequent contexts** rather than **informative** ones (see slides). PMI asks: *do $w$ and $c$ occur together **more often than we would expect by chance**?*

$$\text{PMI}(w, c) = \log \frac{P(w, c)}{P(w)\,P(c)}$$

with probabilities estimated from the count matrix $M$ (let $N = \sum_{w,c} M_{w,c}$ be the total count):

$$P(w,c) = \frac{M_{w,c}}{N}, \qquad P(w) = \frac{\sum_c M_{w,c}}{N} \;\;(\text{row sum}), \qquad P(c) = \frac{\sum_w M_{w,c}}{N} \;\;(\text{column sum})$$

If $w$ and $c$ were independent, $P(w,c) = P(w)P(c)$ and PMI $= \log 1 = 0$. **Positive** PMI: they attract each other. **Negative** PMI: they avoid each other. In practice we use **Positive PMI**:
$$\text{PPMI}(w,c) = \max(0, \text{PMI}(w,c)),$$
and cells with count 0 stay 0 (their PMI would be $\log 0 = -\infty$).

### 💻 Task 2.2 (11 pts) — PPMI
Implement `ppmi(M)` returning a sparse matrix of the same shape. Only compute PMI for the **non-zero** cells (otherwise you build a dense 100M-cell matrix).

**Hints:** `M.tocoo()` gives you `.row`, `.col` and `.data` arrays of the non-zero cells. Row sums: `np.asarray(M.sum(axis=1)).ravel()`. Build the result with `sp.csr_matrix((values, (row, col)), shape=M.shape)` and call `.eliminate_zeros()` on it.

In [ ]:
def ppmi(M):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ check (given)
P_toy = ppmi(M_toy)
print(pd.DataFrame(P_toy.toarray().round(2), index=toy_vocab, columns=toy_vocab))
D = M_toy.toarray(); N = D.sum()
expected = np.log(D[toy_w2i["kitten"], toy_w2i["cute"]] * N / (D.sum(1)[toy_w2i["kitten"]] * D.sum(0)[toy_w2i["cute"]]))
assert abs(P_toy[toy_w2i["kitten"], toy_w2i["cute"]] - max(expected, 0)) < 1e-5
assert P_toy.min() >= 0 and P_toy.shape == M_toy.shape
print("✅ PPMI looks right")

M_ppmi = ppmi(M_counts)
print(f"PPMI matrix: {M_ppmi.nnz:,} non-zero cells")

### A first look (given)
The helper below wraps an embedding matrix: it L2-normalises every row once (so that a dot product between rows **is** the cosine), and lets you get the vector of a word. It works for sparse and dense matrices.

In [ ]:
class WordVectors:
    def __init__(self, name, M):
        self.name = name
        self.M = normalize(M.astype(np.float32))       # each row now has length 1 (zero rows stay 0)
        self.sparse = sp.issparse(self.M)
    def vec(self, word):
        row = self.M[word2id[word]]
        return np.asarray(row.todense()).ravel() if self.sparse else row
    def all_sims(self, q):
        # dot product of every (unit-length) row with vector q
        return np.asarray(self.M @ q).ravel()

models = {
    "counts":   WordVectors("counts", M_counts),
    "PPMI":     WordVectors("PPMI", M_ppmi),
}

### 💻 Task 2.4 (5 pts; counted in Part 4) — nearest neighbours
Implement `nearest_neighbors(wv, word, k)` returning the `k` most similar words (excluding the word itself) as a list of `(word, similarity)` pairs, sorted from most to least similar. Use `wv.vec` and `wv.all_sims`.

*The 5 points for this implementation are awarded under Part 4, Task 4.1; they are not additional points in Part 2.*

In [ ]:
def nearest_neighbors(wv, word, k=8):
    # TODO
    raise NotImplementedError

In [ ]:
# (given) compare the two count-based models
PROBES = ["king", "france", "computer", "car", "music", "germany", "red", "seven", "running", "good"]
def show_neighbors(model_names, probes=PROBES, k=6):
    rows = {}
    for w in probes:
        if w in word2id:
            rows[w] = [", ".join(n for n, _ in nearest_neighbors(models[m], w, k)) for m in model_names]
    return pd.DataFrame.from_dict(rows, orient="index", columns=model_names)
pd.set_option("display.max_colwidth", 80)
show_neighbors(["counts", "PPMI"])

### ✍️ Question 2.5 (4 pts)
Compare the neighbours of the **raw counts** with those of **PPMI**. What kind of words tend to show up as neighbours with raw counts, and why? What changes with PPMI?

**Your answer:**

*(write here)*

### ✍️ Question 2.6 (4 pts)
(a) In words, what does it mean if $\text{PMI}(w,c) > 0$, $= 0$, $< 0$?
(b) Why do we clip negative values to 0 (two reasons are enough: think about how reliable negative values are in a finite corpus, and about the cells with count 0)?
(c) PMI has a known bias: for **rare** context words, a *single* co-occurrence can already give a very high PMI. Why? (Look at the denominator.)

**Your answer:**

*(write here)*

---
## Part 3 — The *predict* approach (33 points)

From the slides, the generic recipe for learning embeddings is:
1. Collect the instances $t_i$ of each word $t$.
2. For each instance, collect its context words $c(t_i)$ (a window).
3. Define a **score function** $\text{score}(t_i, c(t_i); \theta, \mathbf{E})$.
4. Define a **loss** $L$ that is low when the scores are good.
5. **Estimate** $\hat\theta, \hat{\mathbf{E}} = \arg\min_{\theta,\mathbf{E}} L$.
6. Use the estimated $\mathbf{E}$ as your embedding matrix.

The embedding matrix $\mathbf{E}$ starts **random**. Training slowly moves the rows around until words used in similar contexts get similar rows. We implement two scorers from the lecture: **CBOW** and **C&W**.

> 🧭 **A gentle note on training (read this!)**
> Step 5 ("estimate") is done by **gradient descent**: for every parameter (every number in $\mathbf{E}$ and $\theta$), we compute the *gradient*, i.e. in which direction and how strongly a tiny change of that number would change the loss. Then we nudge all parameters a small step in the direction that lowers the loss, on one mini-batch of examples at a time, thousands of times.
> Computing all these gradients efficiently is called **backpropagation**. We will study it in detail in the next lectures and you will implement it yourself in **Assignment 2**. For now PyTorch computes the gradients for us automatically (`loss.backward()`), and we give you the training loop.
>
> **Your job here is the part that defines the model:** the **forward pass** (input → scores) and the **loss** (scores → one number saying how bad the scores are). If those are right, training will work.

### 3.1 Training data: windows

For CBOW the model sees the $2w$ context words around position $n$ and must predict the word at $n$:

$$\underbrace{w_{n-2}\;\; w_{n-1}}_{\text{context}} \;\; \underbrace{w_n}_{\text{target}} \;\; \underbrace{w_{n+1}\;\; w_{n+2}}_{\text{context}}$$

### 💻 Task 3.1 (5 pts)
Implement `make_cbow_examples(ids, window)` returning
- `contexts`: int array of shape `(n_examples, 2*window)` — context ids in the order $w_{n-w}, \dots, w_{n-1}, w_{n+1}, \dots, w_{n+w}$,
- `targets`: int array of shape `(n_examples,)` — the ids of the centre words $w_n$,

for every position `n` that has a full window (`window <= n < len(ids) - window`). Avoid a Python loop over positions: build an array of centre positions with `np.arange` and use fancy indexing `ids[centers + offset]` for each offset.

In [ ]:
def make_cbow_examples(ids, window):
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ check (given)
c, t = make_cbow_examples(np.arange(10, 17), window=2)
print(c, t)
assert c.shape == (3, 4) and list(t) == [12, 13, 14] and list(c[0]) == [10, 11, 13, 14]
print("✅ windows look right")

contexts, targets = make_cbow_examples(ids, WINDOW)
contexts_t = torch.from_numpy(contexts).to(DEVICE)
targets_t = torch.from_numpy(targets).to(DEVICE)
print(f"{len(targets):,} training examples")

### 3.2 CBOW (Mikolov et al. 2013)

From the slides: *embed the context words, add them, project back to vocabulary size, softmax*:

$$P(t_i \mid \text{context}(t_i)) = \text{softmax}\Big(\big(\textstyle\sum_{t_j \in \text{context}(t_i)} \text{onehot}_{t_j}^\top \mathbf{E}\big)\, W_v\Big)$$

Step by step, for one example with context ids $(j_1, j_2, j_3, j_4)$:
1. **Embed:** look up the 4 rows $\mathbf{E}[j_1], \dots, \mathbf{E}[j_4]$, each of size $d$.
2. **Add:** $\mathbf{h} = \sum_k \mathbf{E}[j_k]$ (size $d$) — the "bag" of the context: word order is ignored, hence *Continuous Bag of Words*.
3. **Project:** $\mathbf{l} = \mathbf{h}\,W_v$, one score (*logit*) per vocabulary word (size $|V|$).
4. **Softmax** turns the logits into a probability distribution over the vocabulary.

The loss is the **negative log-likelihood** of the correct target words (averaged over the mini-batch):
$$L = -\frac{1}{B}\sum_{i=1}^{B} \log P(t_i \mid \text{context}(t_i))$$
If the model gives the correct word probability 1, the loss is $-\log 1 = 0$; if it gives it a tiny probability, the loss is large. A model guessing uniformly at random has loss $\log |V| \approx 9.2$ for $|V| = 10{,}000$: that is where training starts.

### 💻 Task 3.2 (10 pts)
Complete `forward` (returns the **logits**, shape `(B, V)`, i.e. *before* the softmax) and `cbow_loss`.
- `self.E(contexts)` looks up the rows for a whole batch at once: shape `(B, 2w, d)`. Sum over the context dimension.
- In `cbow_loss`, use `torch.log_softmax` (numerically safer than `log(softmax(...))`), then pick the log-probability of the correct target for every example. **Do not** use `F.cross_entropy` / `nn.CrossEntropyLoss` (but you may use it to check your result).

In [ ]:
class CBOW(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.E = nn.Embedding(V, d)                 # the embedding matrix E (V x d)
        self.W_v = nn.Linear(d, V, bias=False)      # the projection W_v (d x V)
        nn.init.uniform_(self.E.weight, -0.5 / d, 0.5 / d)

    def forward(self, contexts):
        # contexts: LongTensor (B, 2w)  ->  logits: FloatTensor (B, V)
        # TODO
        raise NotImplementedError


def cbow_loss(logits, targets):
    # logits: (B, V), targets: LongTensor (B,)  ->  scalar: mean negative log-likelihood
    # TODO
    raise NotImplementedError

In [ ]:
# ✅ check (given)
_m = CBOW(50, 8); _c = torch.randint(0, 50, (6, 4)); _t = torch.randint(0, 50, (6,))
_logits = _m(_c)
assert _logits.shape == (6, 50)
assert torch.allclose(cbow_loss(_logits, _t), nn.functional.cross_entropy(_logits, _t), atol=1e-5)
print("✅ CBOW forward and loss look right; loss of an untrained model:", round(cbow_loss(_logits, _t).item(), 3),
      "vs log(V) =", round(math.log(50), 3))

**Training (given).** Watch the loss go down from about $\log |V|$.

In [ ]:
def train(model, batch_loss, n_examples, epochs, lr=5e-3, log_every=500):
    # Generic mini-batch training loop. batch_loss(model, idx) must return the loss on the examples idx.
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, step, t0 = [], 0, time.time()
    for epoch in range(epochs):
        perm = torch.randperm(n_examples, device=DEVICE)           # shuffle the examples
        for start in range(0, n_examples, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            loss = batch_loss(model, idx)      # 1. forward pass + loss   (your code!)
            opt.zero_grad()
            loss.backward()                    # 2. gradients via backpropagation (next lectures)
            opt.step()                         # 3. small step downhill
            history.append(loss.item()); step += 1
            if step % log_every == 0:
                print(f"epoch {epoch+1} step {step:6d}  loss {np.mean(history[-log_every:]):.3f}  ({time.time()-t0:.0f}s)")
    plt.figure(figsize=(6, 3)); plt.plot(np.convolve(history, np.ones(100) / 100, mode="valid"))
    plt.xlabel("step"); plt.ylabel("loss (moving avg)"); plt.title(type(model).__name__); plt.show()
    return model

cbow = CBOW(V_SIZE, DIM).to(DEVICE)
cbow = train(cbow, lambda m, idx: cbow_loss(m(contexts_t[idx]), targets_t[idx]), len(targets), CBOW_EPOCHS)
models["CBOW"] = WordVectors("CBOW", cbow.E.weight.detach().cpu().numpy())
show_neighbors(["PPMI", "CBOW"])

### ✍️ Question 3.3 (3 pts)
(a) The softmax denominator sums over the **whole vocabulary** for **every** training example. What does this mean for training time when $|V|$ grows from 10,000 to 1,000,000 words? Name one trick word2vec uses to avoid it.
(b) After training, CBOW has *two* matrices with one vector per word: $\mathbf{E}$ ($|V| \times d$) and $W_v$ ($d \times |V|$, one column per word). We used $\mathbf{E}$. What is the difference between the two roles?

**Your answer:**

*(write here)*

### 3.3 C&W (Collobert & Weston 2008; Collobert et al. 2011)

A different scoring function. Instead of predicting a word, the network gives a **single score** to a whole window: *how plausible is this piece of text?* From the slides:
- **Embed** the words of the window with $\mathbf{E}$,
- a shallow **convolution** + **MLP** maps them to a **scalar score** $g_{\theta,\mathbf{E}}(s)$.

For a single window of 5 words, the "convolution" is simply a linear layer applied to the **concatenation** of the 5 embeddings, which is what we implement:

$$\mathbf{x} = [\mathbf{E}[w_{n-2}]; \mathbf{E}[w_{n-1}]; \mathbf{E}[w_n]; \mathbf{E}[w_{n+1}]; \mathbf{E}[w_{n+2}]] \in \mathbb{R}^{5d}, \quad \mathbf{h} = \tanh(W_1\mathbf{x} + \mathbf{b}_1), \quad g(s) = \mathbf{w}_2^\top \mathbf{h} + b_2$$

What stops the network from ignoring its input and giving every window a high score (the *useless scorer* from the slides)? For each true window $s$ we create a **distractor** $z$ by **corrupting** it: the centre word is replaced with a random vocabulary word. Then we minimise the **hinge loss**

$$L = \max\big(0,\; 1 - (g(s) - g(z))\big)$$

The loss is 0 once the true window beats its corrupted version by a **margin** of at least 1; otherwise the model is pushed to raise $g(s)$ and lower $g(z)$. Note: no softmax over the vocabulary is needed!

**Which random word?** If we picked replacement words *uniformly* from the vocabulary, most of them would be rare words (remember Question 1.2), and the network could win by just checking "is the centre word frequent?" without looking at the context. So we sample replacements from the **unigram distribution raised to the power 0.75** (`noise_dist`, given below), the same trick word2vec uses for negative sampling: frequent words are picked often, but rare words still get a chance.

### 💻 Task 3.4 (10 pts)
Complete `forward` (returns one score per window, shape `(B,)`), `corrupt` and `hinge_loss`.

In [ ]:
class CWScorer(nn.Module):
    def __init__(self, V, d, window, hidden=100):
        super().__init__()
        self.E = nn.Embedding(V, d)
        self.hidden = nn.Linear((2 * window + 1) * d, hidden)   # W_1, b_1
        self.out = nn.Linear(hidden, 1)                         # w_2, b_2
        nn.init.uniform_(self.E.weight, -0.1, 0.1)

    def forward(self, windows):
        # windows: LongTensor (B, 2w+1)  ->  scores: FloatTensor (B,)
        # TODO: embed, concatenate (hint: .flatten(start_dim=1)), hidden layer + tanh, output layer
        raise NotImplementedError


def corrupt(windows, noise_dist):
    # TODO: return a COPY of windows (B, 2w+1) where the centre word of every window is replaced
    #       by a word id sampled from noise_dist (a probability vector of length V).
    #       Hint: torch.multinomial(noise_dist, num_samples, replacement=True). Do not modify the input!
    raise NotImplementedError


def hinge_loss(score_true, score_corrupt, margin=1.0):
    # TODO: mean over the batch of max(0, margin - (score_true - score_corrupt))
    raise NotImplementedError

In [ ]:
# ✅ check (given)
_m = CWScorer(50, 8, window=2); _w = torch.randint(0, 50, (6, 5))
assert _m(_w).shape == (6,)
_z = corrupt(_w, torch.ones(50) / 50)
assert torch.equal(_z[:, [0, 1, 3, 4]], _w[:, [0, 1, 3, 4]]) and _z is not _w
assert abs(hinge_loss(torch.tensor([3.0, 0.0]), torch.tensor([0.0, 0.5])).item() - 0.75) < 1e-6
print("✅ C&W forward, corruption and hinge loss look right")

In [ ]:
# (given) training: the windows are the CBOW examples with the target put back in the middle
windows_t = torch.cat([contexts_t[:, :WINDOW], targets_t[:, None], contexts_t[:, WINDOW:]], dim=1)

freqs = np.bincount(ids, minlength=V_SIZE) ** 0.75
noise_dist = torch.tensor(freqs / freqs.sum(), dtype=torch.float32, device=DEVICE)

def cw_batch_loss(model, idx):
    s = windows_t[idx]
    return hinge_loss(model(s), model(corrupt(s, noise_dist)))

cw = CWScorer(V_SIZE, DIM, WINDOW).to(DEVICE)
cw = train(cw, cw_batch_loss, len(windows_t), CW_EPOCHS, lr=3e-3)
models["C&W"] = WordVectors("C&W", cw.E.weight.detach().cpu().numpy())
show_neighbors(["PPMI", "CBOW", "C&W"])

### ✍️ Question 3.5 (3 pts)
(a) What would the C&W model learn if we trained it **without** corrupted windows, i.e. just maximising $g(s)$ for real windows?
(b) Why is one C&W training step much cheaper than one CBOW step for a large vocabulary?
(c) In C&W, embeddings are **concatenated**; in CBOW they are **summed**. What information does the sum throw away?

**Your answer:**

*(write here)*

### ✍️ Question 3.6 (2 pts)
Count models and predict models are not as different as they look. Both start from the same information. Which? And what does a predict model do that a count model never does explicitly?

**Your answer:**

*(write here)*

---
## Part 4 — Count vs. Predict: the evaluation (25 points)

We now have four models: `counts`, `PPMI`, `CBOW`, and `C&W`. All were built from the **same corpus** and the **same window**, so the comparison is fair. We test them in three ways.

### Task 4.1 (5 pts; assesses the implementation in Task 2.4): nearest neighbours
Look at the neighbour tables above and below. Feel free to add your own probe words.

In [ ]:
show_neighbors(list(models), probes=["king", "france", "computer", "seven", "running"], k=5)

### 4.2 Correlation with human judgements
Psycholinguists asked people to rate word pairs on a scale, e.g. *car–automobile* (very similar) vs. *car–banana* (unrelated). If our vectors capture meaning, **cosine similarity should go up and down together with the human ratings**. We measure this with the **Spearman rank correlation** $\rho$: it compares the *rankings* of the pairs (1 = identical ranking, 0 = no relation).

We use three datasets:
- **MEN** (Bruni, Tran & Baroni 2014): 3,000 pairs rated for **relatedness** (*sun–sunlight*, *river–water*),
- **WordSim-353** (Finkelstein et al. 2002): 353 pairs, a mix of similarity and relatedness,
- **SimLex-999** (Hill, Reichart & Korhonen 2015): 999 pairs rated for **strict similarity** (*coast–shore* high, but *car–tyre* low even though they are related).

In [ ]:
# (given) load the datasets as lists of (word1, word2, human_score)
def load_men(path):
    df = pd.read_csv(path)
    return [(a.split("-")[0], b.split("-")[0], s) for a, b, s in zip(df.word1, df.word2, df.similarity)]
def load_tab(path):
    out = []
    for line in open(path, encoding="utf8"):
        if line.startswith("#") or not line.strip(): continue
        a, b, s = line.strip().split("\t")[:3]
        out.append((a.lower(), b.lower(), float(s)))
    return out
SIM_DATA = {"MEN": load_men(f"{DATA}/men.csv"),
            "WS353": load_tab(f"{DATA}/wordsim353.tsv"),
            "SimLex": load_tab(f"{DATA}/simlex999.txt")}
for k, v in SIM_DATA.items(): print(k, len(v), v[:3])

### 💻 Task 4.2 (6 pts)
Implement `evaluate_similarity(wv, dataset)`: for every pair where **both** words are in our vocabulary, compute the cosine similarity of their vectors; then return the Spearman correlation between your similarities and the human scores (use `scipy.stats.spearmanr`), and the **coverage** (fraction of pairs you could use).

In [ ]:
def evaluate_similarity(wv, dataset):
    # TODO: return (rho, coverage)
    raise NotImplementedError

In [ ]:
# (given)
sim_results = {name: {ds: evaluate_similarity(wv, data)[0] for ds, data in SIM_DATA.items()} for name, wv in models.items()}
print("coverage:", {ds: f"{evaluate_similarity(models['CBOW'], d)[1]:.0%}" for ds, d in SIM_DATA.items()})
pd.DataFrame(sim_results).T.round(3)

### 4.3 Analogies: king − man + woman = ?
If the vectors encode relations as *directions*, then $\mathbf{king} - \mathbf{man} + \mathbf{woman}$ should land near $\mathbf{queen}$. For an analogy *a : b :: c : ?* (*man : king :: woman : ?*) we compute
$$\mathbf{q} = \mathbf{b} - \mathbf{a} + \mathbf{c}$$
and return the words whose vectors have the highest cosine with $\mathbf{q}$, **excluding** $a$, $b$ and $c$ themselves (this method is called *3CosAdd*).

### 💻 Task 4.3 (5 pts)
Implement `analogy(wv, a, b, c, k)` returning the top-`k` words. (Since all rows of `wv.M` have length 1, `wv.all_sims(q)` gives the cosines up to a constant factor $1/\|\mathbf{q}\|$, which does not change the ranking.)

In [ ]:
def analogy(wv, a, b, c, k=5):
    # TODO: a : b :: c : ?   e.g. analogy(wv, "man", "king", "woman") -> ["queen", ...]
    raise NotImplementedError

In [ ]:
# (given) some classics: try your own too!
TESTS = [("man", "king", "woman"), ("france", "paris", "germany"), ("italy", "rome", "japan"),
         ("good", "better", "bad"), ("walk", "walking", "swim"), ("big", "bigger", "small")]
TESTS = [t for t in TESTS if all(w in word2id for w in t)]
pd.DataFrame({f"{a}:{b} :: {c}:?": {name: ", ".join(analogy(wv, a, b, c, 3)) for name, wv in models.items()}
              for a, b, c in TESTS}).T

In [ ]:
# (given) systematic evaluation on the Google analogy test set (Mikolov et al. 2013), restricted to our vocabulary
def load_analogies(path):
    sem, syn, section = [], [], None
    for line in open(path):
        if line.startswith(":"):
            section = line.strip(); continue
        q = line.lower().split()
        if all(w in word2id for w in q):
            (syn if "gram" in section else sem).append(q)
    return sem, syn
ANALOGIES = dict(zip(["semantic", "syntactic"], load_analogies(f"{DATA}/questions-words.txt")))
rng = np.random.default_rng(SEED)
ANALOGIES = {k: [v[i] for i in rng.permutation(len(v))[:2000]] for k, v in ANALOGIES.items()}  # max 2000 each, for speed
print({k: len(v) for k, v in ANALOGIES.items()})

def analogy_accuracy(wv, questions, batch=500):
    correct = 0
    for s in range(0, len(questions), batch):
        Q = questions[s:s + batch]
        A, B, C, D = ([word2id[q[i]] for q in Q] for i in range(4))
        get = lambda idx: np.asarray(wv.M[idx].todense()) if wv.sparse else wv.M[idx]
        q = get(B) - get(A) + get(C)                       # (batch, dims)
        sims = np.asarray(wv.M @ q.T).T                    # (batch, V)
        rows = np.arange(len(Q))
        for idx in (A, B, C): sims[rows, idx] = -np.inf
        correct += int((sims.argmax(1) == np.array(D)).sum())
    return correct / len(questions)

for name, wv in models.items():
    for kind, qs in ANALOGIES.items():
        sim_results[name][f"analogy-{kind}"] = analogy_accuracy(wv, qs)
results = pd.DataFrame(sim_results).T.round(3)
results

In [ ]:
# (given) a picture: the same words in 2-D (PCA) for a count and a predict model
WORDS = ["france", "paris", "germany", "berlin", "italy", "rome", "spain", "madrid", "japan", "tokyo",
         "king", "queen", "man", "woman", "boy", "girl", "father", "mother",
         "one", "two", "three", "four", "five", "red", "blue", "green", "black", "white"]
WORDS = [w for w in WORDS if w in word2id]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, name in zip(axes, ["PPMI", "CBOW"]):
    X = np.stack([models[name].vec(w) for w in WORDS])
    P = PCA(n_components=2, random_state=SEED).fit_transform(X)
    ax.scatter(P[:, 0], P[:, 1], s=12)
    for (x, y), w in zip(P, WORDS): ax.annotate(w, (x, y), fontsize=9)
    ax.set_title(name)
plt.show()

### ✍️ Question 4.4 (3 pts) — similarity vs. relatedness
Compare the scores on **MEN / WS353** with those on **SimLex**. Why are SimLex scores lower for (almost) all models? Give an example of a word pair that is *related* but not *similar*, and explain why a window-based distributional model will tend to give it a high cosine anyway.

**Your answer:**

*(write here)*

### ✍️ Question 4.5 (2 pts) — analogies
Why must we exclude $a$, $b$ and $c$ from the candidates? (Try it without the exclusion if you are unsure.) Which analogy type (semantic or syntactic) works better for your best model, and can you guess why?

**Your answer:**

*(write here)*

### ✍️ Question 4.6 (4 pts) — the verdict: count or predict?
Use your results table to answer:
(a) Which model is best overall, and is the winner the same for every test? Rank the four models.
(b) The famous paper *"Don't count, predict!"* (Baroni, Dinu & Kruszewski, ACL 2014) found predict models clearly better; a year later Levy, Goldberg & Dagan (TACL 2015) showed that a **well-tuned** count model (with suitable weighting and other improvements) can be competitive. Do your results support "don't count, predict", or rather "it depends on the tuning"? Name one factor in **our** setup that could change the outcome (e.g. corpus size, training epochs, dimensionality, window size).
(c) All four models give **one vector per word type**. Using the example *"I record the record"* from the lecture, explain why this is a limitation, and how the models we will see later in the course (Transformers) address it.

**Your answer:**

*(write here)*

---
## Optional extras (not graded, for the curious)
- **Window size:** rebuild the count model with `WINDOW = 1` and `WINDOW = 5`. Which window works better for SimLex, and which for MEN?
- **Train longer:** increase the number of epochs for CBOW and C&W. Does the ranking of count-based and predictive models change?

### References
- Baroni, Dinu & Kruszewski (2014). *Don't count, predict! A systematic comparison of context-counting vs. context-predicting semantic vectors.* ACL.
- Bruni, Tran & Baroni (2014). *Multimodal distributional semantics.* JAIR (MEN dataset).
- Collobert & Weston (2008); Collobert et al. (2011). *Natural language processing (almost) from scratch.* JMLR.
- Hill, Reichart & Korhonen (2015). *SimLex-999.* Computational Linguistics.
- Levy, Goldberg & Dagan (2015). *Improving distributional similarity with lessons learned from word embeddings.* TACL.
- Mikolov et al. (2013). *Efficient estimation of word representations in vector space.* ICLR workshop.

**Before submitting:** `Runtime → Restart and run all`, check that all outputs are there, save, rename the notebook file using the surname(s) of all group members, and upload the `.ipynb` on Stud.IP (one submission per group). Do not write your names inside the notebook. Good luck and have fun! 🎉